# Preprocessing

## Load Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
train_df=pd.read_csv("../data/train.csv")
train_df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## Handling Missing Values:

In [2]:
missing = train_df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]

missing_ratio = (missing / len(train_df)) * 100
print(missing_ratio.sort_values(ascending=False))

PoolQC          99.520548
MiscFeature     96.301370
Alley           93.767123
Fence           80.753425
MasVnrType      59.726027
FireplaceQu     47.260274
LotFrontage     17.739726
GarageQual       5.547945
GarageFinish     5.547945
GarageType       5.547945
GarageYrBlt      5.547945
GarageCond       5.547945
BsmtFinType2     2.602740
BsmtExposure     2.602740
BsmtCond         2.534247
BsmtQual         2.534247
BsmtFinType1     2.534247
MasVnrArea       0.547945
Electrical       0.068493
dtype: float64


In [3]:
for col in ['PoolQC', 'MiscFeature', 'Alley', 'Fence']:
    train_df[col] = train_df[col].fillna("None")

#### Categorical Missing Data

In [4]:
train_df['MasVnrType'] = train_df['MasVnrType'].fillna("None")
train_df['FireplaceQu'] = train_df['FireplaceQu'].fillna("None")

missing = no masonry veneer / no fireplace quality

#### Numerical Missing Data


In [5]:
train_df['LotFrontage'] = train_df['LotFrontage'].fillna(train_df['LotFrontage'].median())

In [6]:
none_fill = [
    'GarageQual','GarageFinish','GarageType','GarageCond',
    'BsmtFinType1','BsmtFinType2','BsmtQual','BsmtCond','BsmtExposure'
]

for col in none_fill:
    train_df[col] = train_df[col].fillna("None")

#### Numeric garage


In [7]:
train_df['GarageYrBlt'] = train_df['GarageYrBlt'].fillna(0)

In [8]:
train_df['GarageYrBlt'] = train_df['GarageYrBlt'].fillna(train_df['YearBuilt'])

#### Numeric area feature

In [9]:
train_df['MasVnrArea'] = train_df['MasVnrArea'].fillna(0)

#### Small categorical

In [10]:
train_df['Electrical'] = train_df['Electrical'].fillna(train_df['Electrical'].mode()[0])

In [11]:
train_df.isnull().sum().sum()

np.int64(0)

All empty cells are filled!

1. Categorical missing → "None"

means feature does not exist

2. Numeric missing (small %) → median / mode

safe estimation

3. Structural numeric missing → 0 or logical replacement

like GarageYrBlt, MasVnrArea

## Feature Engineering:

In [12]:
train_df["TotalSF"] = train_df["TotalBsmtSF"] + train_df["1stFlrSF"] + train_df["2ndFlrSF"]

## Encoding:

In [13]:
cat_cols = train_df.select_dtypes(include=['object', "string"]).columns
print(cat_cols)

Index(['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',
       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
       'SaleType', 'SaleCondition'],
      dtype='str')


### One-Hot Encoding

In [15]:
train_df=pd.get_dummies(train_df, columns=cat_cols, drop_first=True)

In [16]:
train_df.shape

(1460, 262)

80 cols has transformed into 262 cols!

In [17]:
print(train_df.head())

   Id  MSSubClass  LotFrontage  LotArea  OverallQual  OverallCond  YearBuilt  \
0   1          60         65.0     8450            7            5       2003   
1   2          20         80.0     9600            6            8       1976   
2   3          60         68.0    11250            7            5       2001   
3   4          70         60.0     9550            7            5       1915   
4   5          60         84.0    14260            8            5       2000   

   YearRemodAdd  MasVnrArea  BsmtFinSF1  ...  SaleType_ConLI  SaleType_ConLw  \
0          2003       196.0         706  ...           False           False   
1          1976         0.0         978  ...           False           False   
2          2002       162.0         486  ...           False           False   
3          1970         0.0         216  ...           False           False   
4          2000       350.0         655  ...           False           False   

   SaleType_New  SaleType_Oth  SaleTyp

In [18]:
X=train_df.drop('SalePrice',axis=1)
y=train_df['SalePrice']

In [19]:
processed_train = train_df.copy()